# Simulated data generator (library)

Replays games in Gazebo (teleported pieces), randomizes lighting and camera placement per
session, grabs `/camera/image` after each ply and writes labelled square crops +
`manifest.csv` in exactly the format of `gambit_vision/docs/contracts.md`, so the ResNet-18
can be trained **sim only**, **real only** and **mixed** (README "Updated plan (11/4/2026)").
Sim sessions are named `sim_*` so splits by `session_id` keep sim and real apart.

It also holds the camera's **true intrinsics** (from the SDF) and its exact projection, used
by `run_sim_calibration.ipynb` to grade a simulated ChArUco calibration, and to get exact
ground-truth board corners for testing the homography on simulated frames.

Config: `configs/data_generator.yaml`.

**TODO checklist**
- [ ] true intrinsics K from the camera SDF, exact world -> pixel projection
- [ ] compare an estimated K against the true one
- [ ] session naming, lighting / camera-jitter sampling, quaternion helper
- [ ] manifest rows (64 per frame, contracts.md columns) and CSV writing
- [ ] `apply_lighting` / `apply_camera_pose` backends (gz service light_config / set_pose)
- [ ] `grab_frame` from `/camera/image` (rclpy + cv_bridge, or `gz topic`)
- [ ] `extract_square_crops`: reuse gambit_vision's homography + crop code (M2), do not duplicate
- [ ] run `generate_session` for real and check the crops by eye

In [ ]:
%run ./teleport.ipynb

In [ ]:
import csv
import math
from datetime import date

import numpy as np

DATA_CFG_PATH = REPO / "configs/data_generator.yaml"
with open(DATA_CFG_PATH) as f:
    DATA_CFG = yaml.safe_load(f)

## True intrinsics from the SDF

Gazebo renders an ideal pinhole camera (no distortion, square pixels). With horizontal field of
view `hfov` and image width `w`:

    fx = fy = (w / 2) / tan(hfov / 2),   cx = w / 2,   cy = h / 2

For hfov = 0.95 rad and 640 x 480: fx ~ 622.0 px. (Check against `/camera/camera_info` once;
some Gazebo versions use (w - 1) / 2 for cx.) A simulated calibration should recover these.

In [ ]:
def camera_params_from_sdf(path=CAMERA_SDF):
    """{'hfov': rad, 'width': px, 'height': px} read from the camera model's SDF."""
    # Design: ElementTree; find the <sensor><camera> element and read horizontal_fov,
    # image/width, image/height (findtext + float/int).
    raise NotImplementedError

def true_intrinsics(params=None):
    """3x3 K of the simulated camera (distortion is zero)."""
    # Design: pinhole formula from the markdown above (square pixels -> fy = fx, principal point
    # at the image center). Decision for you: w/2 or (w-1)/2 for cx (check /camera/camera_info).
    raise NotImplementedError

def compare_intrinsics(K_est, K_true=None, dist=None):
    """Errors of an estimated calibration vs the truth: relative focal error, principal point (px)."""
    # Design:
    # - Relative errors for fx, fy ((est - true) / true) and pixel errors for cx, cy, as a dict.
    # - If distortion coefficients are given, report their norm (the truth is all zeros).
    raise NotImplementedError

## Exact projection (ground truth)

SDF pose `x y z roll pitch yaw` -> rotation R = Rz(yaw) Ry(pitch) Rx(roll). The camera link looks
along +x (y left, z up); the optical frame (OpenCV) is X right = -y_link, Y down = -z_link,
Z forward = x_link. A world point p maps to p_link = R^T (p - t), then u = fx X/Z + cx, v = fy Y/Z + cy.

In [ ]:
def rpy_to_matrix(roll, pitch, yaw):
    # Design: elementary rotations about x, y, z composed as Rz(yaw) @ Ry(pitch) @ Rx(roll) (SDF/ROS convention).
    raise NotImplementedError

def rpy_to_quat(roll, pitch, yaw):
    """(x, y, z, w) quaternion for SDF/ROS roll-pitch-yaw."""
    # Design: standard half-angle formula for the same ZYX order as rpy_to_matrix; return (x, y, z, w).
    # Check it by converting the quaternion back to a matrix and comparing with rpy_to_matrix.
    raise NotImplementedError

LINK_TO_OPTICAL = np.array([[0, -1, 0], [0, 0, -1], [1, 0, 0]], dtype=float)

def project_points(pts_world, cam_pose=None, K=None):
    """(N, 3) world points -> (N, 2) pixels for the camera at `cam_pose` (x, y, z, r, p, y)."""
    # Design:
    # - World -> camera link: p_link = R^T (p - t) with R from rpy_to_matrix (vectorize over N).
    # - Link -> optical frame with LINK_TO_OPTICAL; reject points with Z <= 0 (behind the camera).
    # - Perspective divide, then apply K. Defaults: camera_pose() and true_intrinsics().
    # Why: exact ground truth to grade homographies / calibrations on simulated frames.
    raise NotImplementedError

def board_corner_pixels(cam_pose=None, cfg=BOARD):
    """Pixel positions of the playing-area corners (outer corners of a1, h1, h8, a8)."""
    # Design: outer corner = square center pushed half a square outward on both axes (signs per
    # corner, see the world frame in world_generator), z = board top; project the 4 points and
    # return {'a1': (u, v), 'h1': ..., 'h8': ..., 'a8': ...}.
    raise NotImplementedError

## Session randomization

One session = one lighting setup + one camera placement (contracts.md: splits are by session).
Sampling is pure Python; *applying* it to Gazebo is a backend. Lights are changed with
`gz service -s /world/gambit_chess/light_config --reqtype gz.msgs.Light` (UserCommands plugin,
already in the world). The camera is moved with set_pose on the `board_camera` model.

In [ ]:
def session_id(index, lighting, fmt=None):
    # Design: fill sessions.name_format (index, lighting preset, today's date) and insist the
    # result starts with 'sim_' (keeps sim and real sessions apart in the splits).
    raise NotImplementedError

def color_temp_to_rgb(kelvin):
    """Approximate blackbody color (Tanner Helland's fit), channels in 0..1."""
    # Design: piecewise fit of R, G, B vs temperature/100 (look up Tanner Helland's constants),
    # clamp to 0..255 and scale to 0..1. Only needs to be plausible: warm lamp vs cool daylight.
    raise NotImplementedError

def _rotate_dir(d, max_deg, rng):
    # Design: normalize d, pick a random axis perpendicular to it, rotate d by a uniform angle in
    # [0, max_deg] about that axis (Rodrigues; the cross-product term suffices since axis is perpendicular to d).
    raise NotImplementedError

def sample_lighting(rng, cfg=DATA_CFG):
    # Design:
    # - From the `lighting` config: preset name, sun color (color temperature -> RGB) scaled by a
    #   random intensity, sun direction = default (-0.3, -0.2, -1) jittered, ambient, shadows on/off.
    # - Return plain Python floats/bools in a dict (keys used by light_config_req).
    raise NotImplementedError

def light_config_req(light, name="sun"):
    """gz.msgs.Light request text for /world/<world>/light_config."""
    # Design: protobuf text with name, type DIRECTIONAL, diffuse {r g b a}, direction {x y z},
    # cast_shadows (lower-case true/false).
    raise NotImplementedError

def light_config_command(light, world=WORLD_NAME):
    # Design: argv list for `gz service -s /world/<world>/light_config` (reqtype gz.msgs.Light,
    # reptype gz.msgs.Boolean, timeout, --req light_config_req(light)); same shape as SetPose.gz_command.
    raise NotImplementedError

def sample_camera_pose(rng, cfg=DATA_CFG, board=BOARD, per_frame=False):
    """Jittered (x, y, z, roll, pitch, yaw) for board_camera, still looking near the board center."""
    # Design:
    # - Per session: jitter position (camera_jitter.position_m), look_at point (look_at_m) and roll.
    # - Per frame (per_frame=True): only a small position jitter, same look_at, roll 0 (camera shake).
    # - Re-aim with camera_rpy() on the jittered position/look_at so the board stays centered.
    raise NotImplementedError

def camera_set_pose(cam_pose):
    # Design: SetPose for the 'board_camera' model; orientation via rpy_to_quat.
    raise NotImplementedError

def apply_lighting(light, backend):
    """TODO: send light_config_command(light) (gz CLI) or the ROS equivalent; also set the scene
    ambient (no runtime service for scene ambient in Harmonic: either bake it into a world variant
    per session, or approximate with a second, non-shadow-casting light)."""
    raise NotImplementedError("TODO: apply_lighting")

def apply_camera_pose(cam_pose, backend):
    """TODO: backend.send(camera_set_pose(cam_pose)), then wait a frame so the next image uses it.
    Check that set_pose moves a static model's sensor in your Gazebo version."""
    raise NotImplementedError("TODO: apply_camera_pose")

## Capture (TODO)

`grab_frame` must return a fresh (H, W, 3) uint8 RGB image rendered **after** the last set_pose
(wait `capture.settle_s`, then drop one frame). `extract_square_crops` must produce the crops the
same way as for real frames: ArUco detection -> homography -> warp -> crop (owned by gambit_vision
M2; `%run ../../gambit_vision/notebooks/<capture/homography notebook>` and call it). Do not use
the ground-truth corners for crops of the training data, or sim results will be too optimistic;
use `board_corner_pixels()` only to measure the homography error.

In [ ]:
def grab_frame(timeout_s=2.0):
    """TODO: one RGB frame from /camera/image.
    Option A (ROS): rclpy subscriber + cv_bridge imgmsg_to_cv2(msg, 'rgb8') (see ros_playground).
    Option B (no ROS): `gz topic -e -n 1 -t /camera/image` and parse the gz.msgs.Image bytes."""
    raise NotImplementedError("TODO: grab_frame")

def extract_square_crops(frame):
    """TODO: frame -> (warped board image, {square_name: crop (200, 100, 3)}) via gambit_vision M2."""
    raise NotImplementedError("TODO: extract_square_crops")

## Ground-truth labels -> manifest rows

64 rows per frame in square-index order (a1 = 0 ... h8 = 63), label = python-chess symbol
(`P N B R Q K p n b r q k`) or `empty`, matching `labels.CLASSES`. `path` is relative to the
session directory.

In [ ]:
MANIFEST_COLUMNS = ["session_id", "game_id", "move_idx", "frame_id", "square", "label", "path", "fen"]
CLASSES = ["empty", "P", "N", "B", "R", "Q", "K", "p", "n", "b", "r", "q", "k"]   # = labels.CLASSES

def frame_id(counter):
    # Design: zero-padded 6-digit string (sorts correctly as text).
    raise NotImplementedError

def manifest_rows(session, game_id, move_idx, fid, board, ext=".png"):
    # Design:
    # - One dict per square in chess.SQUARES order (a1 first), keys = MANIFEST_COLUMNS.
    # - label = piece.symbol() or 'empty'; path = squares/<fid>_<square><ext> (relative to the
    #   session dir); fen = board.board_fen() (placement only).
    raise NotImplementedError

def write_manifest(rows, session_dir, append=True):
    """Write / append rows to <session_dir>/manifest.csv (header only when the file is new)."""
    # Design: create the directory; csv.DictWriter with MANIFEST_COLUMNS; write the header only
    # when the file is new or append=False; return the CSV path.
    raise NotImplementedError

def save_crops(session_dir, fid, warped, crops):
    """Write frames/<fid>.png and squares/<fid>_<sq>.png (crops are RGB arrays)."""
    # Design: cv2.imwrite expects BGR -> convert with cv2.cvtColor(..., COLOR_RGB2BGR); the warped
    # full board only when capture.save_full_frames is set; create the subfolders as needed.
    raise NotImplementedError

## One session (orchestration)

The loop is plain plumbing over the pieces above; it should fail loudly at the first unfinished piece.

In [ ]:
def generate_session(index, games, backend, out_root=None, cfg=DATA_CFG, seed=0):
    """Replay `games` (list of (game_id, pgn_text)) in one randomized session; returns session dir."""
    # Design:
    # - One RNG per session (seed + index) so sessions are reproducible and independent.
    # - Sample + apply lighting and the session camera pose; session id from the lighting preset.
    # - For each game, pgn_replay() and send each ply's requests (games.max_plies cap); only
    #   capture every games.every_nth_ply ply.
    # - Per captured ply: capture.frames_per_ply frames, each with a small per-frame camera jitter,
    #   grab -> crops -> save -> append 64 manifest rows labelled from the ply's FEN.
    # - Return the session directory (under output.root unless out_root is given).
    raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - true_intrinsics(): fx == 320 / tan(0.475) ~ 622.0, cx == 320, cy == 240
# - compare_intrinsics(K with fx, fy scaled by 1.01) -> fx_rel_err == 0.01, cx_err_px == 0
# - project_points([camera look_at]) == (320, 240) (the camera aims at it)
# - board_corner_pixels(): all four inside the 640 x 480 image; rank 8 is nearer the camera so
#   a8 has larger v than a1; the a-file is on the image right, so u(a1) > u(h1)
# - rpy_to_quat(0.3, -0.2, 1.1) converted back to a matrix equals rpy_to_matrix(0.3, -0.2, 1.1)
# - sample_lighting(rng): preset in the config list, light_config_req contains 'type: DIRECTIONAL'
# - sample_camera_pose(rng): position within camera_jitter.position_m of board camera.position;
#   camera_set_pose(...).name == 'board_camera'
# - session_id(3, 'dim') == 'sim_03_dim'
# - manifest_rows after 1.e4: 64 rows with columns MANIFEST_COLUMNS, e4 'P', e2 'empty', e8 'k',
#   labels subset of CLASSES, rows[28]['path'] == 'squares/000007_e4.png' (frame 7)
# - write_manifest twice into a temp dir -> pandas reads 128 rows with one header
# Example shape of a test:
# K = true_intrinsics(); assert abs(K[0, 0] - 622.0) < 0.5
print("Self-check: not written yet (see plan above)")